# CNN Español Colombia — Bronze → Silver → Gold → Grafo de dependencias

Laboratorio de ingesta y enriquecimiento. Las capas (ver `docs/arquitectura.md` §11):

| Capa | Qué | Quién | ¿LLM? |
|---|---|---|---|
| **BRONZE** | El HTML/RSS crudo tal cual llegó | `CNNColombiaFetcher` (descarga) | No |
| **SILVER** | `CNNArticle` parseado, deduplicado, fechas UTC | `CNNColombiaFetcher` (parsing determinista) | No |
| **GOLD** | 15 tópicos + canal de transmisión FX + entidades | `NewsAnalyzer` del pipeline (`gpt-5-mini` + `with_structured_output`) | **Sí — aquí entra el LLM** |

La taxonomía es **la misma del pipeline** (`cop_fx/contracts.py`) y tiene dos niveles:
- `topic`: qué ES la noticia — incluye `public_health`, `environment_climate`, `sports`, `security_conflict`, `labor_social`...
- `fx_relevance` + `fx_channel`: si transmite al USD/COP y POR QUÉ mecanismo — una sequía es `indirect` vía `inflation`; un partido de fútbol es `none` y **pesa cero en la señal por contrato**.

> La frontera de capa es el momento en que un LLM toca el dato: si mañana mejoras el prompt, reprocesas GOLD desde SILVER sin re-scrapear.

**Prerequisito**: `OPENAI_API_KEY` en el `.env` de la raíz del proyecto (el mismo que usa el pipeline).

---

### Actualizacion de producto: coherencia del adjudicador

El pipeline ya no confia solo en la prosa del LLM adjudicador. El juez debe declarar `dominant_signal` (`news`, `timeseries`, `market`, `none`) y el codigo valida que la direccion final coincida con esa senal. Si el racional dice que domina una noticia fiscal pero la direccion emitida apunta al lado contrario, el sistema corrige la direccion, baja/confina la confianza y deja el motivo en `consistency_notes`.

Los prompts tambien fueron endurecidos: cada agente debe trabajar como analista independiente, rechazar ruido, distinguir sorpresa de repeticion, nombrar canal de transmision FX y explicar por que una senal pierde contra otra.

## 0. Entorno — cargar `.env` y validar claves

In [ ]:
# Setup — limpio: sin os.chdir ni rutas adivinadas por cwd.
# cop_fx.paths resuelve la raíz desde el paquete; Settings lee el .env absoluto.
import sys

try:
    import cop_fx  # noqa: F401
except ModuleNotFoundError:
    raise RuntimeError(
        f"KERNEL EQUIVOCADO ({sys.executable}).\n"
        "En VS Code: selector de kernel → 'Python (cop-fx-intelligence)'."
    ) from None

from cop_fx.config.settings import get_settings
from cop_fx.paths import DATA_DIR, PROJECT_ROOT

settings = get_settings()
assert settings.openai_api_key is not None, f"Falta OPENAI_API_KEY en {PROJECT_ROOT / '.env'}"
print(f"✓ Proyecto: {PROJECT_ROOT.name} · LLM: {settings.llm_model}")

In [ ]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from cop_fx.data.cnn_fetcher import CNNArticle, CNNColombiaFetcher
from cop_fx.paths import DATA_DIR

DB_PATH = DATA_DIR / "cnn_articles.db"   # ruta absoluta del paquete, no del cwd
DB_PATH.parent.mkdir(exist_ok=True)
print(f"Base de datos: {DB_PATH}")

## 1. BRONZE → SILVER — Scraping CNN Colombia

El fetcher descarga el feed/HTML crudo (Bronze) y lo parsea de forma **determinista** a `CNNArticle` (Silver): sin LLM y sin grafo, puro parsing. En la Fase 1 del roadmap el crudo se persistirá a disco *antes* de parsear, para poder reprocesar sin re-scrapear.

In [ ]:
fetcher = CNNColombiaFetcher(max_articles=100)

# enrich_authors=True → hace 1 request por artículo para obtener el autor real
articles: list[CNNArticle] = fetcher.fetch(enrich_authors=True)

print(f"\nArtículos obtenidos : {len(articles)}")
print(f"Con autor           : {sum(1 for a in articles if a.author)}")

In [ ]:
df_raw = pd.DataFrame([
    {
        "fecha"  : a.published_at.strftime("%Y-%m-%d"),
        "título" : a.title,
        "autor"  : a.author or "(sin autor)",
        "url"    : a.url,
    }
    for a in articles
])
df_raw

## 2. GOLD — Análisis enriquecido con el `NewsAnalyzer` del pipeline

Una sola llamada estructurada produce, **por artículo**: `topic` (15 dominios), `keywords`, `entities` (personas/instituciones/empresas/lugares), `fx_relevance` (direct / indirect / none), `fx_channel` (tasas, inflación, términos de intercambio, riesgo país, flujos, crecimiento), `severity`, `bullish_cop` y `reasoning` — más la narrativa FX del día.

El contrato Pydantic garantiza coherencia: si `fx_relevance="none"`, el validador fuerza `fx_channel="none"` y `severity="low"`. Una noticia de deportes no puede contaminar la señal ni aunque el modelo se equivoque.

## Keywords, entidades y taxonomía normalizada

La capa GOLD no solo clasifica `topic`, `fx_channel` y severidad. También extrae `keywords` y `entities`, que sirven para auditar qué conceptos están dominando la señal noticiosa. En producto, el dashboard pondera estas palabras por importancia (`severity x relevance + canal`) para evitar que una palabra repetida en noticias de baja relevancia parezca más importante que un shock material.

La normalización de topics vive en `cop_fx.analysis.topic_taxonomy`: agrupa dominios finos en familias de decisión (`Riesgo pais`, `Riesgo fiscal`, `Commodities`, etc.) y marca huecos de investigación cuando una noticia material cae en `other`, no tiene canal FX, o parece un falso positivo.


In [ ]:
from cop_fx.analysis.topic_taxonomy import article_importance, normalize_topic

if "df" in globals() and {"topic", "keywords", "fx_relevance", "fx_channel", "severity"}.issubset(df.columns):
    df_keywords = df.copy()
    df_keywords["topic_family"] = df_keywords["topic"].apply(normalize_topic)
    df_keywords["importance"] = df_keywords.apply(article_importance, axis=1)
    display(df_keywords[["title", "topic_family", "keywords", "entities", "importance"]].head(20))
else:
    print("Ejecuta primero la celda que construye df GOLD para ver keywords ponderadas.")


In [ ]:
from cop_fx.analysis.news_analyzer import NewsAnalyzer
from cop_fx.data.article_body import attach_bodies

# El MISMO analyzer del pipeline — y sobre la NOTICIA COMPLETA, no el titular:
# attach_bodies descarga el cuerpo de cada artículo antes de clasificar.
attach_bodies(articles)
analyzer = NewsAnalyzer()
analysis = analyzer.analyze(articles)

con_cuerpo = sum(1 for a in articles if getattr(a, "body", ""))
print(f"✓ {len(analysis.items)} artículos analizados · {con_cuerpo} con cuerpo completo\n")
print(f"Narrativa FX del día:\n{analysis.narrative}")

In [ ]:
df = pd.DataFrame([
    {
        "title"       : it.article.title,
        "author"      : it.article.author,
        "published_at": it.article.published_at.isoformat(),
        "summary"     : it.article.summary,
        "url"         : it.article.url,
        "source"      : it.article.source,
        "topic"       : it.topic,
        "keywords"    : json.dumps(it.keywords, ensure_ascii=False),
        "entities"    : json.dumps(it.entities, ensure_ascii=False),
        "fx_relevance": it.fx_relevance,
        "fx_channel"  : it.fx_channel,
        "severity"    : it.severity,
        "bullish_cop" : int(it.bullish_cop),
        "reasoning"   : it.reasoning,
        "fetched_at"  : datetime.now(tz=timezone.utc).isoformat(),
    }
    for it in analysis.items
])
print(f"✓ {len(df)} filas GOLD")
df[["title", "topic", "fx_relevance", "fx_channel", "severity", "bullish_cop"]]

In [ ]:
# ¿Cuánta señal FX hay realmente en un día de CNN Colombia?
# Esta tabla ES el argumento del router (Etapa 2): la mayoría del día es ruido FX.
pd.crosstab(df["topic"], df["fx_relevance"], margins=True)

## 3. Grafo de dependencias — tópicos ↔ entidades

Grafo bipartito: cada tópico se conecta con las entidades que sus noticias mencionan; el peso de la arista es el número de co-menciones. Sobre él corremos **detección de comunidades** (greedy modularity).

Por qué importa: las comunidades son los **clusters naturales del día** — exactamente la unidad de trabajo que el orchestrator de la Etapa 3 despacha con `Send` (un worker por cluster). Si una entidad puentea dos tópicos (p. ej. *Petro* aparece en `political_risk` y en `fiscal_policy`), eso es señal de una historia transversal que un análisis por-artículo no ve.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

G = nx.Graph()
for it in analysis.items:
    t = f"◼ {it.topic}"
    G.add_node(t, kind="topic")
    for e in it.entities:
        en = f"● {e}"
        G.add_node(en, kind="entity")
        w = (G.get_edge_data(t, en) or {}).get("weight", 0)
        G.add_edge(t, en, weight=w + 1)

communities = list(greedy_modularity_communities(G, weight="weight"))
print(f"Nodos: {G.number_of_nodes()} | Aristas: {G.number_of_edges()} | Comunidades: {len(communities)}")

# Entidades-puente: tocan más de un tópico → historias transversales
bridges = [
    n for n in G.nodes
    if G.nodes[n]["kind"] == "entity"
    and len({nb for nb in G.neighbors(n) if G.nodes[nb]["kind"] == "topic"}) > 1
]
print(f"Entidades-puente entre tópicos: {bridges}")

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.65, seed=42, weight="weight")
palette = plt.cm.tab10.colors
node_color = [
    palette[next(ci for ci, c in enumerate(communities) if n in c) % 10] for n in G.nodes
]
sizes = [
    900 + 250 * G.degree(n) if G.nodes[n]["kind"] == "topic" else 120 + 90 * G.degree(n)
    for n in G.nodes
]
nx.draw_networkx_edges(G, pos, alpha=0.3, width=[G[u][v]["weight"] for u, v in G.edges], ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_color, node_size=sizes, alpha=0.85, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title("Grafo de dependencias: tópicos (◼) ↔ entidades (●) — color = comunidad")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Señal direccional por cluster de tópico — el mismo cálculo determinista
# que aggregate_signals ejecuta dentro del grafo (cero LLM).
from cop_fx.contracts import ArticleAnalysis, aggregate_news_signal


def to_contract(items: list) -> list[ArticleAnalysis]:
    return [
        ArticleAnalysis(
            index=i,
            topic=it.topic,
            keywords=it.keywords or ["sin_clasificar"],
            entities=it.entities,
            fx_relevance=it.fx_relevance,
            fx_channel=it.fx_channel,
            severity=it.severity,
            bullish_cop=it.bullish_cop,
            reasoning=it.reasoning[:240],
        )
        for i, it in enumerate(items)
    ]


clusters: dict[str, list] = {}
for it in analysis.items:
    clusters.setdefault(it.topic, []).append(it)

rows = []
for topic, items in sorted(clusters.items(), key=lambda kv: -len(kv[1])):
    sig = aggregate_news_signal(to_contract(items))
    rows.append({
        "cluster": topic,
        "artículos": len(items),
        "score": sig.score,
        "dirección": sig.direction,
    })

global_sig = aggregate_news_signal(
    to_contract(analysis.items),
    {i: it.article.title for i, it in enumerate(analysis.items)},
)
print(f"SEÑAL GLOBAL: {global_sig.direction} (score={global_sig.score})")
print(f"Drivers: {global_sig.drivers}\n")
pd.DataFrame(rows)

## 4. En vivo: la rama de noticias del grafo real (Etapas 2 + 3)

Este es el subgrafo de noticias del pipeline (`src/cop_fx/agents/graph.py`) con sus **nodos reales**, corriendo sobre los artículos de hoy:

1. `check_materiality` — una sola llamada decide si hay señal FX **y de paso etiqueta cada titular** (esos tags siembran los clusters, reutilizando el mismo paso).
2. Router condicional: día sin señal → `skip_news` (no se ejecuta más análisis); día con señal → `orchestrate`.
3. `orchestrate` + **`Send`** — fan-out dinámico: un `topic_worker` por cluster, todos en paralelo en el mismo superstep.
4. `aggregate_signals` — consolida los workers en una señal direccional, determinista.

Esto es el patrón **orchestrator-workers**: la cantidad de workers la decide el dato del día, no el grafo — la API `Send` los emite en runtime.

In [ ]:
from langgraph.graph import END, START, StateGraph

from cop_fx.agents import nodes as N
from cop_fx.agents.state import PipelineState

g = StateGraph(PipelineState)
g.add_node("check_materiality", N.check_materiality)
g.add_node("skip_news", N.skip_news)
g.add_node("orchestrate", N.orchestrate)
g.add_node("topic_worker", N.topic_worker)
g.add_node("aggregate_signals", N.aggregate_signals)

g.add_edge(START, "check_materiality")
g.add_conditional_edges(
    "check_materiality",
    N.route_materiality,
    {"orchestrate": "orchestrate", "skip_news": "skip_news"},
)
g.add_conditional_edges("orchestrate", N.fan_out_clusters, ["topic_worker"])
g.add_edge("topic_worker", "aggregate_signals")
g.add_edge("aggregate_signals", END)
g.add_edge("skip_news", END)

news_branch = g.compile()
print(news_branch.get_graph().draw_mermaid())

result = news_branch.invoke({"raw_articles": articles, "errors": []})

print(f"¿Material?: {result.get('has_material_news')} — {result.get('materiality_reason')}\n")
print(f"Clusters despachados: { {t: len(i) for t, i in result.get('clusters', {}).items()} }\n")
print(f"Señal de noticias: {result.get('news_signal')}\n")
print("Narrativas por cluster:")
print(result.get("news_summary", ""))

## 5. Base de datos SQLite — upsert con el esquema enriquecido

El esquema GOLD creció (`entities`, `fx_relevance`, `fx_channel`, `severity`, `bullish_cop`, `reasoning`). Dos detalles de ingeniería:

- **Migración liviana**: si la tabla existe con el esquema viejo, se agregan las columnas faltantes con `ALTER TABLE` (sin perder datos).
- **Upsert por URL**: si el artículo ya existía, se **refresca su enriquecimiento** — así puedes mejorar el prompt y reprocesar GOLD sobre los mismos artículos.

In [ ]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from cop_fx.data.cnn_fetcher import CNNArticle, CNNColombiaFetcher
from cop_fx.paths import DATA_DIR

DB_PATH = DATA_DIR / "cnn_articles.db"   # ruta absoluta del paquete, no del cwd
DB_PATH.parent.mkdir(exist_ok=True)
print(f"Base de datos: {DB_PATH}")

## 4. Consultar la base de datos

In [ ]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from cop_fx.data.cnn_fetcher import CNNArticle, CNNColombiaFetcher
from cop_fx.paths import DATA_DIR

DB_PATH = DATA_DIR / "cnn_articles.db"   # ruta absoluta del paquete, no del cwd
DB_PATH.parent.mkdir(exist_ok=True)
print(f"Base de datos: {DB_PATH}")

In [ ]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from cop_fx.data.cnn_fetcher import CNNArticle, CNNColombiaFetcher
from cop_fx.paths import DATA_DIR

DB_PATH = DATA_DIR / "cnn_articles.db"   # ruta absoluta del paquete, no del cwd
DB_PATH.parent.mkdir(exist_ok=True)
print(f"Base de datos: {DB_PATH}")

## 5. Exportar

In [ ]:
export_path = ROOT / "data"

# CSV
csv_file = export_path / "cnn_articles.csv"
df_db.to_csv(csv_file, index=False)
print(f"✓ CSV  → {csv_file}")

# Parquet (más eficiente para subir a cloud / BigQuery)
parquet_file = export_path / "cnn_articles.parquet"
df_db.to_parquet(parquet_file, index=False)
print(f"✓ Parquet → {parquet_file}")

## Co-ocurrencia pairwise de keywords

Inspirado en el flujo `tidytext` (`unnest_tokens` → `pairwise_cor`), este proyecto usa una versión Python sobre la capa GOLD: cada noticia es un documento, cada keyword juzgada es una variable booleana, y cada par recibe coeficiente phi. Antes de correlacionar, se eliminan stopwords de dominio y el LLM decide si el término es accionable para USD/COP. Esto evita que palabras como `Colombia` dominen por frecuencia sin aportar señal.


In [ ]:
from cop_fx.analysis.keyword_analysis import (
    build_term_documents,
    judge_terms,
    pairwise_phi,
    rank_terms,
)
from cop_fx.config.settings import get_settings

settings = get_settings()

if "df_keywords" in globals():
    gold_terms = build_term_documents(
        df_keywords[df_keywords["fx_relevance"] != "none"],
        domain_stopwords=settings.keyword_domain_stopwords,
        include_entities=False,
    )
    candidates = (
        gold_terms.groupby(["term_key", "term"], as_index=False)
        .agg(score=("importance", "sum"), noticias=("doc_id", "nunique"))
        .sort_values(["score", "noticias"], ascending=False)
        .head(settings.keyword_top_n * 2)["term"]
        .tolist()
    )
    decisions = judge_terms(
        candidates,
        use_llm=settings.keyword_llm_judge_enabled,
        domain_stopwords=settings.keyword_domain_stopwords,
    )
    ranked_terms = rank_terms(
        gold_terms,
        decisions,
        min_articles=settings.keyword_min_articles,
        top_n=settings.keyword_top_n,
    )
    pairs = pairwise_phi(
        gold_terms,
        ranked_terms["termino"].tolist(),
        min_joint=settings.keyword_pairwise_min_joint,
    )
    display(ranked_terms[["termino", "score", "noticias", "llm_score", "reason"]].head(15))
    display(pairs.head(15))
else:
    print("Ejecuta primero la sección de keywords para construir df_keywords.")


## Conclusiones — qué significa cada artefacto

Recorriste la frontera de capas del *medallion* y lo que cada pieza significa:

- **Bronze → Silver** es determinista a propósito: separar el crudo del parseado
  te deja reprocesar sin volver a scrapear. La frontera con **Gold** es el punto
  exacto donde un LLM toca el dato (clasificación estructurada con
  `with_structured_output`); por eso topic/keywords son Gold, no Bronze.
- **Taxonomía de dos niveles**: `topic` dice *qué ES* la noticia; `fx_channel`
  dice *cómo transmite* al USD/COP. Separarlos es lo que convierte "opinión" en
  "mecanismo". `fx_relevance="none"` pesa **cero por contrato** — el ruido no
  contamina la señal aunque el modelo se equivoque.
- **Importancia = severidad × relevancia (+ bonus de canal/tópico)**: una métrica
  aritmética, no una corazonada. Evita que una palabra repetida en noticias
  irrelevantes parezca un shock.
- **Grafo bipartito tópico↔entidad + comunidades**: las comunidades son los
  *clusters del día*, exactamente la unidad que el orquestador despacha con
  `Send`. Una entidad que puentea dos tópicos delata una historia transversal.
- **Co-ocurrencia φ (phi)**: mide si dos términos aparecen juntos *más de lo que
  el azar predice* (no solo frecuencia). Útil entre varias noticias; con una sola
  mención, el par es anecdótico.

> Para replicar la técnica: un único nodo `with_structured_output` sobre un lote
> de artículos ya te da la capa Gold. El fan-out por cluster y el adjudicador se
> montan encima cuando una sola llamada deja de alcanzar.

## 🎓 Proyecto individual — dónde engancha en esta notebook

Esta notebook trabaja la cadena de noticias (Bronze→Silver→Gold) y el análisis
por artículo — justo el terreno de dos de las pistas del proyecto:

- **Pista 1 (ReAct):** el clasificador que ves aquí *adivina* la severidad. El reto
  es darle herramientas (Brent/DXY/TRM) para que la **confirme** con datos.
- **Pista 3 (Orchestrator-workers):** hoy todos los clusters se analizan con el
  mismo prompt; el reto es asignar una **persona** por tipo de cluster.

Consigna completa, retos y rúbrica: [`docs/proyecto_individual.md`](../docs/proyecto_individual.md).